# Análisis de Performance Comercial — Olist Marketplace (2016–2018)

**Autor:** [Nombre del Analista]  
**Fecha:** [Fecha del Análisis]  
**Dataset:** Olist Brazilian E-Commerce — [Kaggle](https://www.kaggle.com/datasets/olistbr/brazilian-ecommerce)

---

## Resumen Ejecutivo

[Completar con los hallazgos principales del análisis al finalizar el reporte]

---

## Contexto del Negocio

Olist es una startup brasileña fundada en 2015 que opera como intermediario entre pequeños comercios y los grandes marketplaces de e-commerce del país (Mercado Libre, B2W, Via Varejo, entre otros). El modelo permite a vendedores individuales publicar su catálogo en múltiples plataformas mediante un único contrato, con logística y servicio al cliente centralizados.

El dataset cubre el período septiembre 2016 – octubre 2018 e incluye datos reales de transacciones, ciclos de entrega y reseñas de clientes. La base de datos comprende:

- **pedidos**: ~99,000 órdenes con timestamps del ciclo de vida completo
- **clientes**: ~99,000 compradores distribuidos por estados brasileños
- **items_pedido**: ~113,000 líneas de producto con precio y flete por vendedor
- **pagos**: ~104,000 registros con método de pago, cuotas y monto
- **resenas**: ~98,000 evaluaciones (puntuación 1–5) con timestamps de respuesta
- **productos**: ~33,000 SKUs con categoría, dimensiones y peso
- **vendedores**: ~3,100 vendedores activos en múltiples estados

---

## Configuración del Entorno

Conexión a la base de datos y verificación del esquema disponible antes de iniciar el análisis.

In [3]:
# Importar librerias necesarias
import sqlite3
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os

# Configuracion de visualizacion
plt.style.use("seaborn-v0_8-whitegrid")
sns.set_palette("Set2")
plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["font.size"] = 11

# Conexion a la base de datos
DB_PATH = "data/portafolio_olist.db"

if not os.path.exists(DB_PATH):
    raise FileNotFoundError(
        f"No se encontro la base de datos en '{DB_PATH}'. "
        "Ejecutar primero: python data/lab1-generacion-datasets.py"
    )

conn = sqlite3.connect(DB_PATH)


def q(query):
    """Ejecuta una consulta SQL y retorna el resultado como DataFrame."""
    return pd.read_sql_query(query, conn)


# Verificar tablas disponibles
print("Tablas disponibles:")
print(q("SELECT name FROM sqlite_master WHERE type='table' ORDER BY name;").to_string(index=False))


Tablas disponibles:
        name
    clientes
items_pedido
       pagos
     pedidos
   productos
     resenas
  vendedores


---

## 1. Caracterización del Período de Análisis

### 1.1 Resumen Operacional

Antes de cualquier segmentación, un reporte ejecutivo requiere establecer la escala de la operación. La tabla pedidos registra cada orden procesada, pagos consolida el valor monetario de cada transacción en el campo valor_pago, clientes almacena la identidad única del comprador en cliente_unico_id, e items_pedido vincula cada orden con el vendedor que la atendió. Cruzando estas cuatro tablas se obtienen los denominadores de todo el análisis: volumen total, revenue bruto, cobertura de clientes y tamaño activo de la red de vendedores.


### 1.2 Evolución Mensual de Pedidos y Revenue

El e-commerce brasileño muestra estacionalidad marcada por el Black Friday en noviembre y las fiestas de fin de año. La tabla pedidos registra la fecha de cada orden en fecha_compra, y la tabla pagos acumula el valor de cada transacción en valor_pago. Agrupando ambas por mes se construye la serie temporal que permite identificar picos de demanda, meses de contracción y la tendencia de crecimiento general durante el período 2016–2018.


### 1.3 Distribución por Estado del Pedido

El campo estado en la tabla pedidos registra la fase del ciclo de vida de cada orden: delivered, shipped, canceled, unavailable, invoiced, processing, created y approved. Una distribución operacionalmente sana concentra la mayoría de órdenes en delivered; acumulaciones en estados intermedios señalan cuellos de botella en el proceso. La tasa de cancelación —pedidos en estado canceled sobre el total— es un indicador directo de fricción en la experiencia de compra.


---

## 2. Red de Vendedores

### 2.1 Top 15 Vendedores por Revenue

En modelos de marketplace, el revenue tiende a concentrarse en un grupo pequeño de operadores. La tabla items_pedido registra cada línea de venta con su precio y flete por vendedor, y la tabla vendedores almacena el estado (unidad federativa) de cada uno. Cruzando ambas con pedidos —filtrado a estado = delivered— se obtiene el ranking de los 15 mayores aportantes al revenue y la proporción del total que concentran, un indicador clave de dependencia operacional.


### 2.2 Distribución Geográfica de Vendedores

La ubicación del vendedor determina los tiempos y costos de entrega. La tabla vendedores registra el estado brasileño de cada operador, e items_pedido contiene el precio y flete de cada transacción. Cruzando ambas se cuantifica cómo se distribuye la red de vendedores por región y si la concentración geográfica se corresponde con la concentración de revenue —es decir, si la desigualdad territorial también es una desigualdad comercial.


### 2.3 Volumen de Ventas y Satisfacción

La hipótesis es que los vendedores con mayor volumen tienen procesos más maduros y, por tanto, mejor satisfacción. La tabla items_pedido registra cuántos ítems vendió cada operador, y la tabla resenas almacena la puntuación (escala 1–5) que los clientes asignaron a cada pedido en el campo puntuacion. Vinculando ambas vía pedido_id se contrasta volumen y calidad percibida, restringiéndose a vendedores con al menos 30 reseñas para garantizar representatividad estadística.


---

## 3. Satisfacción del Cliente

### 3.1 Distribución de Puntuaciones

Las reseñas capturan la percepción del cliente sobre todo el proceso, desde la descripción del producto hasta la entrega. La tabla resenas registra la evaluación en el campo puntuacion (escala 1–5) y la fecha en que el vendedor respondió en fecha_respuesta. La distribución de puntuaciones revela el perfil de satisfacción general, y la tasa de respuesta del vendedor —calculada sobre reseñas con fecha_respuesta no nula— es un indicador secundario de la calidad del servicio post-venta.


### 3.2 Satisfacción por Categoría de Producto

La categoría del producto determina la complejidad logística y el riesgo de incidencias. La tabla productos almacena la categoría de cada SKU en el campo categoria, y la tabla resenas registra la puntuacion por pedido. Ambas se vinculan a través de la cadena resenas → pedidos → items_pedido → productos. El análisis compara la puntuación promedio y la tasa de insatisfacción severa (puntuacion ≤ 2) entre categorías con al menos 100 reseñas, identificando los segmentos que concentran los mayores problemas de calidad percibida.


### 3.3 Impacto del Tiempo de Entrega en la Satisfacción

El cumplimiento de la fecha prometida es el principal driver de satisfacción en e-commerce. La tabla pedidos registra dos fechas clave: fecha_estimada_entrega, que es la promesa comunicada al cliente al momento de la compra, y fecha_entrega_cliente, que registra cuándo el pedido llegó efectivamente. La diferencia entre ambas —en días— clasifica cada orden como anticipada, puntual o con retraso, y se contrasta con la puntuacion en resenas para cuantificar el costo en satisfacción de no cumplir el plazo.


---

## 4. Eficiencia Logística

### 4.1 Tasa Global de Entregas a Tiempo (OTD Rate)

El On-Time Delivery Rate mide el porcentaje de pedidos entregados en o antes de la fecha prometida al cliente, y es el KPI logístico de referencia del sector. La tabla pedidos registra la fecha de compra en fecha_compra, la entrega real en fecha_entrega_cliente y el plazo comprometido en fecha_estimada_entrega. Comparando las dos últimas sobre el universo de pedidos con estado = delivered se obtiene el OTD Rate global, el tiempo promedio puerta a puerta y los promedios de adelanto y retraso para cada grupo.


### 4.2 Performance Logística por Estado del Vendedor

La distancia geográfica entre vendedor y cliente es el principal determinante estructural del tiempo de entrega. La tabla vendedores registra el estado de origen de cada operador, items_pedido contiene el flete por transacción, y pedidos almacena las fechas del ciclo de vida. Cruzando estas tres fuentes se desglosa la performance logística por región de despacho: tiempo de entrega promedio, OTD Rate y costo de flete promedio para cada unidad federativa.


### 4.3 Categorías con Mayor Tasa de Retraso

Ciertas categorías presentan sistemáticamente mayor incidencia de retrasos, por características físicas del producto, complejidad de preparación del envío o distribución geográfica de sus vendedores. La tabla productos almacena la categoria de cada SKU, y pedidos registra la fecha de entrega real y la estimada. Cruzando ambas se calcula, por categoría, la tasa de retraso y el promedio de días de demora en las órdenes que no cumplieron el plazo, sobre categorías con al menos 50 pedidos entregados.


---

## 5. Categorías y Métodos de Pago

### 5.1 Top Categorías por Revenue

La distribución de revenue por categoría determina la arquitectura comercial del catálogo y orienta las inversiones en desarrollo de oferta. La tabla productos clasifica cada SKU en una categoria, e items_pedido registra el precio y flete de cada línea de venta. Cruzando ambas sobre pedidos entregados se obtiene el ranking de las 15 categorías con mayor aporte al revenue total y la participación porcentual de cada una, diferenciando entre categorías de alto valor unitario y categorías masivas de bajo ticket.


### 5.2 Estructura de Costos por Categoría

El ratio flete/precio mide la carga logística sobre el valor de la transacción. En categorías de bajo precio unitario, el costo de envío puede representar una fracción significativa del total, generando fricción en la conversión y reduciendo el margen efectivo del vendedor. La tabla items_pedido registra el precio y el flete de cada línea de venta, y la tabla productos permite agrupar por categoria. El análisis reporta ese ratio para las 20 categorías con mayor volumen de ventas.


### 5.3 Métodos de Pago y Uso de Cuotas

El comportamiento de pago en Brasil combina boleto bancário, débito y crédito con parcelamento, práctica cultural arraigada incluso en compras de montos moderados. La tabla pagos registra el tipo_pago utilizado en cada transacción, el valor en valor_pago y el número de cuotas en cuotas. Agrupando por tipo_pago se obtiene la distribución de volumen y valor por método; analizando la distribución de cuotas para tarjeta de crédito se cuantifica el alcance del financiamiento a plazo en la base de compradores.


---

## 6. Síntesis y Recomendaciones Estratégicas

Esta sección consolida los hallazgos clave del análisis y formula recomendaciones accionables para las áreas funcionales relevantes. La síntesis debe basarse directamente en los valores cuantitativos obtenidos en las secciones anteriores.

### 6.1 Hallazgos Principales

Completar con los insights más relevantes identificados a lo largo del análisis:

**Escala y Tendencias del Negocio:**  
[Período cubierto, volumen total de pedidos y revenue, tendencias mensuales, meses pico]

**Estructura de la Red de Vendedores:**  
[Concentración de revenue (top N vendedores = X% del total), distribución geográfica, relación volumen-satisfacción]

**Satisfacción del Cliente:**  
[Puntuación promedio global, categorías mejor y peor valoradas, impacto cuantificado del retraso en la puntuación]

**Eficiencia Logística:**  
[OTD Rate global, tiempo promedio de entrega, estados con mejor y peor performance, categorías con mayor tasa de retraso]

**Categorías y Pagos:**  
[Top categorías por revenue, categorías con ratio flete/precio elevado, distribución de métodos de pago]

---

### 6.2 Recomendaciones por Área Funcional

**Operaciones y Logística:**  
[Basado en análisis de OTD Rate, tiempos por estado y categorías problemáticas]

**Gestión de Vendedores (Seller Success):**  
[Basado en concentración de revenue, distribución geográfica y relación volumen-satisfacción]

**Producto y Catálogo:**  
[Basado en análisis de categorías, ticket promedio y ratio flete/precio]

**Experiencia del Cliente:**  
[Basado en distribución de reseñas, impacto del retraso, categorías con alta insatisfacción]

**Finanzas y Procesamiento de Pagos:**  
[Basado en distribución de métodos y uso de cuotas]

---

## Apéndice: Consultas SQL de Referencia

Documentar las principales consultas SQL utilizadas en el análisis para garantizar la trazabilidad y reproducibilidad del reporte. Cada entrada debe incluir el propósito de la consulta y el hallazgo clave obtenido.

In [ ]:
# Cerrar conexion a la base de datos
conn.close()
print("Analisis completado — conexion cerrada")
